In [50]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import ast

def extract_float_from_tensor_string(tensor_str):
    """Extract float value from 'tensor(x.x)' string format"""
    match = re.findall(r'tensor\(([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\)', str(tensor_str))
    return [float(x) for x in match]

def extract_activation_values(activation_str):
    """Extract all float values from activation string like '[tensor(1.2), tensor(3.4)]'"""
    try:
        # Clean the string and extract all tensor values
        return extract_float_from_tensor_string(activation_str)
    except:
        return []

def process_cluster_file(filepath):
    """Process a single cluster CSV file and return unit-level statistics"""
    df = pd.read_csv(filepath)
    
    unit_stats = {}
    rang = 0
    for idx, row in df.iterrows():
        unit = row['unit']
        activation_vals = extract_activation_values(row['activation_value_for_samples'])
        rang += (activation_vals[1] - activation_vals[0])
        if activation_vals:
            unit_stats[unit] = {
                'min': min(activation_vals),
                'max': max(activation_vals),
                'avg': np.mean(activation_vals)
                
            }
    unit_stats['avg_range'] = rang/len(df)
    
    
    return unit_stats

   


def analyze_directory_structure(base_path,model):
    """Analyze the complete directory structure and extract all data"""
    
    results = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    
    for pruned_folder in os.listdir(base_path):
        if '%Pruned' not in pruned_folder:
            continue

        # Extract pruning percentage
        pruning_pct = float(pruned_folder.split('%')[0])
        pruned_path = os.path.join(base_path, pruned_folder)

        # Process each cluster file
        for cluster_file in os.listdir(pruned_path):
            if cluster_file.endswith('.csv') and 'Cluster' in cluster_file:
                cluster_name = cluster_file.replace('.csv', '')
                filepath = os.path.join(pruned_path, cluster_file)

                unit_stats = process_cluster_file(filepath)
                results[model][pruning_pct][cluster_name] = unit_stats
    
    return results

def compute_cluster_statistics(results):
    """Compute average min and max activation values for each cluster"""
    
    cluster_stats = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    
    for model in results:
        for pruning_pct in results[model]:
            for cluster in results[model][pruning_pct]:
                unit_stats = results[model][pruning_pct][cluster]
                
                if unit_stats:
                    mins = [stats['min'] for stats in unit_stats.values() if not isinstance(stats, float)]
                    maxs = [stats['max'] for stats in unit_stats.values() if not isinstance(stats, float)]
                
                    
                    cluster_stats[model][pruning_pct][cluster] = {
                        'avg_min': np.mean(mins),
                        'avg_max': np.mean(maxs),
                        'std_min': np.std(mins),
                        'std_max': np.std(maxs),
                    }
                cluster_stats[model][pruning_pct][cluster].update({'range': unit_stats['avg_range']})
                
                        
    
    return cluster_stats

def track_neuron_changes(results):
    """Track how individual neurons' min/max activations change across pruning levels"""
    
    changes = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    
    for model in results:
        pruning_pcts = sorted(results[model].keys())
        
        for cluster in results[model][pruning_pcts[0]]:
            prev_pct = pruning_pcts[0]
            prev_units = results[model][prev_pct].get(cluster, {})
            for i in range(len(pruning_pcts) - 1):
                
                curr_pct = pruning_pcts[i + 1]
                
                
                curr_units = results[model][curr_pct].get(cluster, {})
                
                # Track changes for common units
                common_units = set(prev_units.keys()) & set(curr_units.keys())
                
                min_up = 0
                min_down = 0
                max_up = 0
                max_down = 0
                
                for unit in common_units:
                    if  isinstance(unit, str): continue
                    min_change = curr_units[unit]['min'] - prev_units[unit]['min']
                    max_change = curr_units[unit]['max'] - prev_units[unit]['max']
                    
                    if min_change > 0:
                        min_up += 1
                    elif min_change < 0:
                        min_down += 1
                    
                    if max_change > 0:
                        max_up += 1
                    elif max_change < 0:
                        max_down += 1
                if len(common_units)==0: 
                    changes[model][f"{prev_pct}->{curr_pct}"][cluster] = {
                        'min_up': 0,
                        'min_down': 0,
                        'max_up':0,
                        'max_down':0
                    }
                else:
                    
                    changes[model][f"{prev_pct}->{curr_pct}"][cluster] = {
                        'min_up': min_up/len(common_units),
                        'min_down': min_down/len(common_units),
                        'max_up': max_up/len(common_units),
                        'max_down': max_down/len(common_units)
                    }
    
    return changes
def create_activation_tables(cluster_stats, output_prefix='activation_table'):
    """
    Create tables with clusters on rows and pruning percentages on columns
    for both average min and average max activations.
    
    Parameters:
    -----------
    cluster_stats : dict
        Nested dictionary from compute_cluster_statistics function
    output_prefix : str
        Prefix for output CSV files
    
    Returns:
    --------
    dict : Dictionary of DataFrames for each model and metric
    """
    
    tables = {}
    
    for model in cluster_stats:
        data = cluster_stats[model]
        pruning_pcts = sorted(data.keys())
        clusters = sorted(list(set([c for pct in data for c in data[pct]])))
        
        # Create table for average min activations
        min_data = []
        for cluster in clusters:
            row = [data[pct][cluster]['avg_min'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            min_data.append(row)
        
        df_min = pd.DataFrame(
            min_data,
            index=clusters,
            columns=[f'{pct}%' for pct in pruning_pcts]
        )
        df_min.index.name = 'Cluster'
        
        # Create table for average max activations
        max_data = []
        for cluster in clusters:
            row = [data[pct][cluster]['avg_max'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            max_data.append(row)
        
        df_max = pd.DataFrame(
            max_data,
            index=clusters,
            columns=[f'{pct}%' for pct in pruning_pcts]
        )
        df_max.index.name = 'Cluster'
        
        range_data = []
        for cluster in clusters:
            row = [data[pct][cluster]['range'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            range_data.append(row)
        
        df_range = pd.DataFrame(
            range_data,
            index=clusters,
            columns=[f'{pct}%' for pct in pruning_pcts]
        )
        df_range.index.name = 'Cluster'
        
        # Store tables
        tables[f'{model}_avg_min'] = df_min
        tables[f'{model}_avg_max'] = df_max
        tables[f'{model}_range'] = df_range
        
        # Save to CSV
        
        print(f"\n{model} - Average Min Activations:")
        print(df_min.to_string(float_format='%.4f'))
        
        print(f"\n{model} - Average Max Activations:")
        print(df_max.to_string(float_format='%.4f'))
        
        print(f"\n{model} - Range Activations:")
        print(df_range.to_string(float_format='%.4f'))
    
    return tables

def plot_cluster_activations(cluster_stats, output_path='cluster_activations.png'):
    """Plot average min and max activation values for each cluster"""
    
    models = list(cluster_stats.keys())
    n_models = len(models)
    
    fig, axes = plt.subplots(n_models, 3, figsize=(15, 5 * n_models))
    if n_models == 1:
        axes = axes.reshape(1, -1)
    
    for idx, model in enumerate(models):
        data = cluster_stats[model]
        pruning_pcts = sorted(data.keys())
        clusters = sorted(list(set([c for pct in data for c in data[pct]])))
        
        # Plot avg min activations
        ax_min = axes[idx, 0]
        for cluster in clusters:
            mins = [data[pct][cluster]['avg_min'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            ax_min.plot(pruning_pcts, mins, marker='o', label=cluster)
        
        ax_min.set_xlabel('Pruning Percentage')
        ax_min.set_ylabel('Average Min Activation')
        ax_min.set_title(f'{model} - Average Min Activations')
        ax_min.legend()
        ax_min.grid(True, alpha=0.3)
        
        # Plot avg max activations
        ax_max = axes[idx, 1]
        for cluster in clusters:
            maxs = [data[pct][cluster]['avg_max'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            ax_max.plot(pruning_pcts, maxs, marker='o', label=cluster)
        
        ax_max.set_xlabel('Pruning Percentage')
        ax_max.set_ylabel('Average Max Activation')
        ax_max.set_title(f'{model} - Average Max Activations')
        ax_max.legend()
        ax_max.grid(True, alpha=0.3)
        
        ax_range = axes[idx, 2]
        print(clusters)
        for cluster in clusters:
            ranges = [data[pct][cluster]['range'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            print(ranges)
            ax_range.plot(pruning_pcts, ranges, marker='o', label=cluster)
        
        ax_range.set_xlabel('Pruning Percentage')
        ax_range.set_ylabel('Average Range')
        ax_range.set_title(f'{model} - Average Range')
        ax_range.legend()
        ax_range.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_neuron_changes(changes, output_path='neuron_changes.png'):
    """Plot number of neurons with increasing/decreasing min/max activations per cluster"""
    
    models = list(changes.keys())
    n_models = len(models)
    
    fig, axes = plt.subplots(n_models, 2, figsize=(16, 5 * n_models))
    if n_models == 1:
        axes = axes.reshape(1, -1)
    
    for idx, model in enumerate(models):
        transitions = sorted(changes[model].keys())
        
        # Get all clusters
        all_clusters = set()
        for transition in transitions:
            all_clusters.update(changes[model][transition].keys())
        clusters = sorted(all_clusters)
        
        # Prepare data per cluster
        cluster_data_min = defaultdict(lambda: {'up': [], 'down': []})
        cluster_data_max = defaultdict(lambda: {'up': [], 'down': []})
        
        for transition in transitions:
            for cluster in clusters:
                if cluster in changes[model][transition]:
                    cluster_data_min[cluster]['up'].append(
                        changes[model][transition][cluster]['min_up'])
                    cluster_data_min[cluster]['down'].append(
                        changes[model][transition][cluster]['min_down'])
                    cluster_data_max[cluster]['up'].append(
                        changes[model][transition][cluster]['max_up'])
                    cluster_data_max[cluster]['down'].append(
                        changes[model][transition][cluster]['max_down'])
                else:
                    cluster_data_min[cluster]['up'].append(0)
                    cluster_data_min[cluster]['down'].append(0)
                    cluster_data_max[cluster]['up'].append(0)
                    cluster_data_max[cluster]['down'].append(0)
        
        # Plot min activation changes
        ax_min = axes[idx, 0]
        x = np.arange(len(transitions))
        width = 0.8 / (len(clusters) * 2)  # Adjust width based on number of clusters
        
        for i, cluster in enumerate(clusters):
            offset_up = (i * 2 - len(clusters)) * width
            offset_down = ((i * 2 + 1) - len(clusters)) * width
            
            ax_min.bar(x + offset_up, cluster_data_min[cluster]['up'], width, 
                      label=f'{cluster} Up', alpha=0.7)
            ax_min.bar(x + offset_down, cluster_data_min[cluster]['down'], width, 
                      label=f'{cluster} Down', alpha=0.7, hatch='//')
        
        ax_min.set_xlabel('Pruning Transition')
        ax_min.set_ylabel('Number of Neurons')
        ax_min.set_title(f'{model} - Min Activation Changes by Cluster')
        ax_min.set_xticks(x)
        ax_min.set_xticklabels(transitions, rotation=45, ha='right')
        ax_min.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax_min.grid(True, alpha=0.3, axis='y')
        
        # Plot max activation changes
        ax_max = axes[idx, 1]
        
        for i, cluster in enumerate(clusters):
            offset_up = (i * 2 - len(clusters)) * width
            offset_down = ((i * 2 + 1) - len(clusters)) * width
            
            ax_max.bar(x + offset_up, cluster_data_max[cluster]['up'], width, 
                      label=f'{cluster} Up', alpha=0.7)
            ax_max.bar(x + offset_down, cluster_data_max[cluster]['down'], width, 
                      label=f'{cluster} Down', alpha=0.7, hatch='//')
        
        ax_max.set_xlabel('Pruning Transition')
        ax_max.set_ylabel('Number of Neurons')
        ax_max.set_title(f'{model} - Max Activation Changes by Cluster')
        ax_max.set_xticks(x)
        ax_max.set_xticklabels(transitions, rotation=45, ha='right')
        ax_max.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax_max.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
def main(base_path,model):
    """Main analysis pipeline"""
    
    print("Analyzing directory structure...")
    results = analyze_directory_structure(base_path,model)
    
    print("Computing cluster statistics...")
    cluster_stats = compute_cluster_statistics(results)
    
    print("Tracking neuron changes...")
    changes = track_neuron_changes(results)
    
    print("Generating plots...")
    #plot_cluster_activations(cluster_stats)
    #plot_neuron_changes(changes)
    create_activation_tables(cluster_stats)
    print("Analysis complete!")
    
    return results, cluster_stats, changes

if __name__ == "__main__":
    # Set your base directory path here
    models=['BERT', 'BOWMAN', 'LLAMA']
    methods=['wanda', 'lottery_ticket']
    for model in models:
        for method in methods:
            base_path = f"/workspace/CCE_NLI/{model}/exp/{method}/Run0.25/Expls"

            results, cluster_stats, changes = main(base_path, f'{model}-{method}')

    # Example: Print summary for one model
    if results:
        model = list(results.keys())[0]
        print(f"\nSummary for {model}:")
        for pruning_pct in sorted(results[model].keys()):
            print(f"  {pruning_pct}% Pruned: {len(results[model][pruning_pct])} clusters")

Analyzing directory structure...
Computing cluster statistics...
Tracking neuron changes...
Generating plots...

BERT-wanda - Average Min Activations:
                    0.0%  25.0%  43.75%  57.812%  68.359%  76.27%
Cluster                                                          
Cluster1IOUS1024N 0.0003 0.0003  0.0003   0.0003   0.0003  0.0002
Cluster2IOUS1024N 0.9000 0.8899  0.8556   0.8047   0.7351  0.6535
Cluster3IOUS1024N 1.9552 1.9267  1.8667   1.7563   1.6156  1.4359

BERT-wanda - Average Max Activations:
                    0.0%  25.0%  43.75%  57.812%  68.359%  76.27%
Cluster                                                          
Cluster1IOUS1024N 0.8990 0.8890  0.8546   0.8038   0.7343  0.6528
Cluster2IOUS1024N 2.0702 2.0477  1.9683   1.8482   1.6849  1.4949
Cluster3IOUS1024N 5.0966 5.0254  4.8653   4.5772   4.2283  3.7435

BERT-wanda - Range Activations:
                    0.0%  25.0%  43.75%  57.812%  68.359%  76.27%
Cluster                                            